In [1]:
import os
import yaml
import copy
import time
import numpy as np
import pandas as pd
import xarray as xr

In [2]:
from xclim.indices import (
    standardized_precipitation_evapotranspiration_index,
    water_budget,
)

from xclim.indices.stats import (
    standardized_index_fit_params, 
    standardized_index
)

In [3]:
import warnings
warnings.filterwarnings("ignore", message="Converting a CFTimeIndex.*noleap.*")

In [4]:
from scipy.interpolate import griddata
from scipy.spatial import Delaunay
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator

In [5]:
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
# fn_ERA5 = f'/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/ERA5_{year}.zarr'
# ds_ERA5 = xr.open_zarr(fn_ERA5)
# fn_CESM = f'/glade/derecho/scratch/ksha/EPRI_data/CESM2_SMYLE/SMYLE_{year-1}-11-01_daily_ensemble.zarr'
# ds_CESM = xr.open_zarr(fn_CESM)

### Preparing gridded yearly metrics

In [8]:
dict_loc = {
    'Pituffik': (76.4, -68.575),
    'Fairbanks': (64.75, -147.4),
    'Guam': (13.475, 144.75),
    'Yuma_PG': (33.125, -114.125),
    'Fort_Bragg': (35.05, -79.115),
}
keys = list(dict_loc.keys())

### Pituffik: melting degree days

In [9]:
key = 'Pituffik'
dir_stn = f'/glade/derecho/scratch/ksha/EPRI_data/METRICS/{key}/'
base_dir = '/glade/derecho/scratch/ksha/EPRI_data/ERA5_grid/'

In [23]:
list_MDD = []
thres = 273.15
for year in range(1958, 2026):

    # get data and variable
    fn_ERA5 = base_dir + f'/{key}/ERA5_{key}_{year}.zarr'
    ds_ERA5 = xr.open_zarr(fn_ERA5)[["2m_temperature"]]
    ds_ERA5 = ds_ERA5.rename({"2m_temperature": "TREFHT"})
    
    ds_ERA5 = ds_ERA5.sel(time=slice(f"{year}-03-01", f"{year}-06-30"))
    mdd_mj = (ds_ERA5["TREFHT"] - thres).clip(min=0).sum(dim="time")
    ds_MDD = mdd_mj.to_dataset(name="MDD").assign_coords(year=year).expand_dims("year")
    
    list_MDD.append(ds_MDD)

ds_MDD_all = xr.merge(list_MDD)

In [35]:
# ds_MDD_all['MDD'].isel(year=3).values

In [24]:
save_name = dir_stn + 'ERA5_MDD.zarr'
# ds_MDD_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/ERA5_MDD.zarr


### Pituffik: Freeze-Thaw days

In [29]:
list_FT = []
Tf = 273.15  # K

for year in range(1958, 2026):

    fn = base_dir + f'/{key}/ERA5_{key}_{year}.zarr'
    ds = xr.open_zarr(fn)[[
        "maximum_2m_temperature_since_previous_post_processing",
        "minimum_2m_temperature_since_previous_post_processing"
    ]]
    
    ds = ds.rename(
        {
            "maximum_2m_temperature_since_previous_post_processing": "TREFHTMX", 
            "minimum_2m_temperature_since_previous_post_processing": "TREFHTMN"
        }
    )

    # Freeze–Thaw day mask (1 if crossing, else 0)
    ft_day = ((ds["TREFHTMN"] < Tf) & (ds["TREFHTMX"] > Tf)).astype("int8")

    # Count days in the season
    ft_count = ft_day.sum(dim="time")

    ds_FT = ft_count.to_dataset(name="FT_days").assign_coords(year=year).expand_dims("year")
    list_FT.append(ds_FT)

ds_FT_all = xr.concat(list_FT, dim="year")

In [37]:
# ds_FT_all['FT_days'].isel(year=3).values

In [31]:
save_name = dir_stn + 'ERA5_FT.zarr'
# ds_FT_all.to_zarr(save_name, mode='w')
print(save_name)

/glade/derecho/scratch/ksha/EPRI_data/METRICS/Pituffik/ERA5_FT.zarr
